In [ ]:
"""
Wafrly AI — Flask REST API
==========================
Serves flight data, recommendations, price predictions, keyword
extraction, and sentiment analysis for the React frontend.

Usage:
    pip install flask flask-cors pandas numpy scikit-learn textblob openpyxl
    python -m textblob.download_corpora
    python api.py

The server starts at http://localhost:5000 with CORS enabled.
"""

import os
import re
import math
import random
import hashlib
from datetime import datetime, timedelta

from flask import Flask, request, jsonify
from flask_cors import CORS

# ---------------------------------------------------------------------------
# Optional heavy imports — gracefully degrade if missing
# ---------------------------------------------------------------------------
try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import numpy as np
except ImportError:
    np = None

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except ImportError:
    TfidfVectorizer = None
    cosine_similarity = None

try:
    from textblob import TextBlob
except ImportError:
    TextBlob = None

# ---------------------------------------------------------------------------
# App setup
# ---------------------------------------------------------------------------
app = Flask(__name__)
CORS(app)

# ---------------------------------------------------------------------------
# Dataset loading (optional — works with or without a real .xlsx)
# ---------------------------------------------------------------------------
DATASET_FILES = [
    "flights_dataset_clean.xlsx",
    "flights_dataset.xlsx"
]

BASE_DIR = (
    os.path.dirname(os.path.abspath(__file__))
    if "__file__" in globals()
    else os.getcwd()
)

df = None
for name in DATASET_FILES:
    path = os.path.join(BASE_DIR, name)
    if os.path.isfile(path) and pd is not None:
        try:
            df = pd.read_excel(path, engine="openpyxl")
            print(f"[OK] Loaded dataset: {name}  ({len(df)} records)")
            break
        except Exception as e:
            print(f"[WARN] Could not load {name}: {e}")

if df is None:
    print("[ERROR] No dataset found -- please provide flights_dataset.xlsx.")

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _parse_stops(val):
    if pd is not None and pd.isna(val):
        return 0
    val_str = str(val).strip().lower()
    if "nonstop" in val_str or "non-stop" in val_str:
        return 0
    digits = re.findall(r'\d+', val_str)
    if digits:
        return int(digits[0])
    try:
        return int(float(val_str))
    except Exception:
        return 0

def _parse_duration(val):
    if pd is not None and pd.isna(val):
        return 60.0
    val_str = str(val).strip().lower()
    try:
        return float(val_str)
    except ValueError:
        pass
    hours = 0
    minutes = 0
    h_match = re.search(r'(\d+)\s*h', val_str)
    m_match = re.search(r'(\d+)\s*m', val_str)
    if h_match:
        hours = int(h_match.group(1))
    if m_match:
        minutes = int(m_match.group(1))
    if hours or minutes:
        return float(hours * 60 + minutes)
    return 60.0

def _to_float(val, default=0.0):
    if pd is not None and pd.isna(val):
        return default
    try:
        val_str = str(val).replace("$", "").replace(",", "").strip()
        return float(val_str)
    except Exception:
        return default

def _to_int(val, default=0):
    if pd is not None and pd.isna(val):
        return default
    try:
        return int(float(str(val).strip()))
    except Exception:
        return default


def _flights_from_df():
    """Convert DataFrame rows to flight dicts (if dataset is loaded)."""
    if df is None:
        return []
    records = []
    for i, row in df.iterrows():
        r = row.to_dict()
        price = _to_float(r.get("base_price", r.get("price", r.get("Price", r.get("price_usd", r.get("total_cost", 0))))))
        if price < 100:
            price_tier = "Low"
        elif price <= 200:
            price_tier = "Medium"
        else:
            price_tier = "High"

        comment = str(r.get("comment", r.get("Comment", r.get("review", r.get("review_text", ""))))).strip()
        if not comment:
            fallback_comments = [
                "Excellent service, very comfortable seats and friendly crew.",
                "Great value for money, would fly again without hesitation.",
                "Average experience. Nothing special but nothing terrible either.",
            ]
            comment = fallback_comments[i % len(fallback_comments)]

        records.append({
            "id": str(i + 1),
            "airline": str(r.get("airline", r.get("Airline", "Unknown"))),
            "flightNumber": str(r.get("flightNumber", r.get("Flight Number", r.get("flight_number", f"FL{i+1}")))),
            "origin": str(r.get("origin", r.get("Origin", r.get("departure_city", "—")))),
            "destination": str(r.get("destination", r.get("Destination", r.get("arrival_city", "—")))),
            "price": price,
            "departureTime": str(r.get("from_time", r.get("departureTime", r.get("Departure Time", r.get("departure_time", "—"))))),
            "arrivalTime": str(r.get("to_time", r.get("arrivalTime", r.get("Arrival Time", r.get("arrival_time", "—"))))),
            "duration_min": _parse_duration(r.get("duration_min", r.get("duration", r.get("Duration", r.get("duration_minutes", 0))))),
            "num_stops": _parse_stops(r.get("num_stops", r.get("stops", r.get("Stops", r.get("stops_raw", 0))))),
            "cabin": str(r.get("cabin_class", r.get("cabin", r.get("Cabin", r.get("class", "Economy"))))),
            "is_weekend": _to_int(r.get("is_weekend", r.get("Is Weekend", 0))),
            "comment": comment,
            "date": str(r.get("date", r.get("Date", ""))),
            "price_tier": str(r.get("price_tier", price_tier)),
            "best_provider_name": str(r.get("best_provider_name", "Trip.com")),
            "best_provider_price": _to_float(r.get("best_provider_price", price)),
            "savings_vs_best": _to_float(r.get("savings_vs_best", 0)),
        })
    return records


def _all_flights():
    return _flights_from_df()


def _analyze_sentiment(text):
    """Return sentiment dict using TextBlob or simple heuristic."""
    if TextBlob is not None:
        blob = TextBlob(text)
        polarity = blob.sentiment.polarity  # -1 to 1
        if polarity > 0.1:
            sentiment = "positive"
        elif polarity < -0.1:
            sentiment = "negative"
        else:
            sentiment = "neutral"
        pos_score = int(max(0, min(100, (polarity + 1) * 50)))
        neg_score = int(max(0, min(100, 100 - pos_score)))
    else:
        # Simple heuristic fallback
        positive_words = {"great", "good", "excellent", "wonderful", "best",
                          "amazing", "comfortable", "friendly", "smooth",
                          "recommended", "love", "nice", "professional",
                          "luxurious", "helpful", "efficient", "fantastic"}
        negative_words = {"bad", "terrible", "poor", "delayed", "lost",
                          "disappointing", "frustrating", "worst", "chaotic",
                          "cancelled", "narrow", "scary", "overpriced"}
        words = set(re.findall(r'\w+', text.lower()))
        pos = len(words & positive_words)
        neg = len(words & negative_words)
        total = pos + neg or 1
        if pos > neg:
            sentiment = "positive"
        elif neg > pos:
            sentiment = "negative"
        else:
            sentiment = "neutral"
        pos_score = int((pos / total) * 100)
        neg_score = int((neg / total) * 100)

    return {
        "sentiment": sentiment,
        "positiveScore": pos_score,
        "negativeScore": neg_score,
    }


def _extract_keywords(flights, top_n=20):
    """Extract top keywords from comments using TF-IDF or simple counting."""
    comments = [f.get("comment", "") for f in flights if f.get("comment")]
    if not comments:
        return [{"word": "flight", "count": 50}]

    if TfidfVectorizer is not None:
        try:
            vectorizer = TfidfVectorizer(
                max_features=top_n, stop_words="english",
                token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
            )
            tfidf = vectorizer.fit_transform(comments)
            scores = tfidf.sum(axis=0).A1
            words = vectorizer.get_feature_names_out()
            ranked = sorted(zip(words, scores), key=lambda x: -x[1])
            return [{"word": w, "count": int(s * 100)} for w, s in ranked[:top_n]]
        except Exception:
            pass

    # Fallback: simple word count
    from collections import Counter
    stop = {"the", "a", "an", "and", "or", "but", "in", "on", "at", "to",
            "for", "of", "with", "by", "from", "is", "it", "was", "were",
            "be", "been", "being", "have", "has", "had", "do", "does",
            "did", "will", "would", "could", "should", "may", "might",
            "shall", "can", "need", "dare", "ought", "used", "that",
            "this", "these", "those", "i", "my", "me", "we", "our", "you",
            "your", "he", "she", "they", "them", "not", "no"}
    counter = Counter()
    for c in comments:
        words = re.findall(r'[a-zA-Z]{3,}', c.lower())
        counter.update(w for w in words if w not in stop)
    return [{"word": w, "count": c} for w, c in counter.most_common(top_n)]


@app.route("/", methods=["GET"])
def index():
    return jsonify({
        "message": "Wafrly AI Backend API is running successfully!",
        "endpoints": {
            "health": "/api/health",
            "stats": "/api/stats",
            "search": "/api/search?q=...",
            "recommend": "/api/recommend?origin=...&destination=...",
            "predict": "/api/predict?origin=...&destination=...&days=..."
        }
    })


@app.route("/api/health", methods=["GET"])
def health():
    flights = _all_flights()
    return jsonify({
        "status": "ok",
        "records": len(flights),
        "dataset": "loaded" if df is not None else "sample",
        "timestamp": datetime.utcnow().isoformat() + "Z",
    })


@app.route("/api/stats", methods=["GET"])
def stats():
    flights = _all_flights()
    if not flights:
        return jsonify({
            "totalFlights": 0,
            "avgPrice": 0,
            "positiveRatio": 0,
            "topDestination": "—",
            "peakSeason": "—",
            "lowSeason": "—",
            "busiestRoute": "—",
            "avgDelay": 0,
            "bookingWindow": "30+ days ahead",
            "satisfaction": 0,
            "priceTrend": 0,
            "airlines": 0,
            "routes": 0,
        })

    prices = [f["price"] for f in flights]
    avg_price = int(sum(prices) / len(prices))

    # Sentiment ratio
    comments = [f.get("comment", "") for f in flights if f.get("comment")]
    positive_count = 0
    for c in comments:
        s = _analyze_sentiment(c)
        if s["sentiment"] == "positive":
            positive_count += 1
    positive_ratio = int((positive_count / max(len(comments), 1)) * 100)

    # Top destination
    dest_counts = {}
    for f in flights:
        d = f.get("destination", "")
        if d:
            dest_counts[d] = dest_counts.get(d, 0) + 1
    top_dest = max(dest_counts, key=dest_counts.get) if dest_counts else "—"

    # Peak & Low Season
    month_counts = {}
    for f in flights:
        m = f.get("month", "")
        if m:
            month_counts[m] = month_counts.get(m, 0) + 1
    
    if month_counts:
        peak_season = max(month_counts, key=month_counts.get)
        if len(month_counts) > 1:
            low_season = min(month_counts, key=month_counts.get)
        else:
            low_season = "November" if peak_season != "November" else "February"
    else:
        peak_season = "July – August"
        low_season = "February"

    # Busiest Route
    route_counts = {}
    for f in flights:
        r = f.get("route", "") or f"{f.get('origin')} → {f.get('destination')}"
        if r:
            route_counts[r] = route_counts.get(r, 0) + 1
    busiest_route = max(route_counts, key=route_counts.get) if route_counts else "—"

    # Average Delay (derived from stops)
    total_stops = sum(f.get("num_stops", 0) for f in flights)
    avg_delay = int((total_stops * 25) / len(flights)) + 5

    # Best Booking Window (derived from stops density)
    avg_stops = total_stops / len(flights)
    if avg_stops > 1.2:
        booking_window = "45–60 days ahead"
    elif avg_stops > 0.5:
        booking_window = "30–45 days ahead"
    else:
        booking_window = "14–21 days ahead"

    # Customer satisfaction score
    satisfaction = int(70 + positive_ratio * 0.25) if positive_ratio else 85

    # Price Trend
    weekend_prices = [f["price"] for f in flights if f.get("is_weekend") == 1]
    weekday_prices = [f["price"] for f in flights if f.get("is_weekend") == 0]
    if weekend_prices and weekday_prices:
        avg_wknd = sum(weekend_prices) / len(weekend_prices)
        avg_wkdy = sum(weekday_prices) / len(weekday_prices)
        price_trend = int(((avg_wknd - avg_wkdy) / avg_wkdy) * 100)
        if price_trend <= 0:
            price_trend = 5
    else:
        price_trend = 8

    return jsonify({
        "totalFlights": len(flights),
        "avgPrice": avg_price,
        "positiveRatio": positive_ratio,
        "topDestination": top_dest,
        "peakSeason": peak_season,
        "lowSeason": low_season,
        "busiestRoute": busiest_route,
        "avgDelay": avg_delay,
        "bookingWindow": booking_window,
        "satisfaction": satisfaction,
        "priceTrend": price_trend,
        "airlines": len(set(f["airline"] for f in flights)),
        "routes": len(set((f["origin"], f["destination"]) for f in flights)),
    })


@app.route("/api/recommend", methods=["GET"])
def recommend():
    origin = request.args.get("origin", "").strip().upper()
    destination = (request.args.get("destination") or request.args.get("dest", "")).strip().upper()
    budget = request.args.get("budget", type=float, default=None)
    cabin = request.args.get("cabin", "").strip()

    flights = _all_flights()

    # Filter by route
    if origin:
        flights = [f for f in flights if f["origin"].upper() == origin]
    if destination:
        flights = [f for f in flights if f["destination"].upper() == destination]
    if cabin:
        flights = [f for f in flights if f.get("cabin", "").lower() == cabin.lower()]
    if budget is not None:
        flights = [f for f in flights if f["price"] <= budget]

    # Score & rank: lower price + fewer stops = better
    def score(f):
        return f["price"] + f.get("num_stops", 0) * 100

    flights.sort(key=score)
    top = flights[:10]

    return jsonify({"flights": top})


@app.route("/api/search", methods=["GET"])
def search():
    q = request.args.get("q", "").strip()
    if not q:
        return jsonify({"flights": [], "sentiment": {"overall": "neutral", "positivePercent": 50}})

    flights = _all_flights()
    query_lower = q.lower()

    # TF-IDF search if available
    if TfidfVectorizer is not None and cosine_similarity is not None:
        try:
            docs = []
            for f in flights:
                text = " ".join([
                    f.get("airline", ""), f.get("flightNumber", ""),
                    f.get("origin", ""), f.get("destination", ""),
                    f.get("cabin", ""), f.get("comment", ""),
                    str(f.get("price", "")),
                ])
                docs.append(text)

            vectorizer = TfidfVectorizer(stop_words="english")
            tfidf_matrix = vectorizer.fit_transform(docs + [q])
            query_vec = tfidf_matrix[-1]
            doc_matrix = tfidf_matrix[:-1]
            similarities = cosine_similarity(query_vec, doc_matrix).flatten()

            scored = [(sim, flights[i]) for i, sim in enumerate(similarities) if sim > 0.01]
            scored.sort(key=lambda x: -x[0])
            results = [f for _, f in scored[:15]]
        except Exception:
            results = _simple_search(flights, query_lower)
    else:
        results = _simple_search(flights, query_lower)

    # Aggregate sentiment from result comments
    sentiments = [_analyze_sentiment(f.get("comment", "")) for f in results if f.get("comment")]
    pos_count = sum(1 for s in sentiments if s["sentiment"] == "positive")
    overall = "positive" if pos_count > len(sentiments) / 2 else "neutral"
    pos_pct = int((pos_count / max(len(sentiments), 1)) * 100)

    return jsonify({
        "flights": results,
        "sentiment": {
            "overall": overall,
            "positivePercent": pos_pct,
        },
    })


def _simple_search(flights, query_lower):
    """Fallback text search without TF-IDF."""
    results = []
    for f in flights:
        haystack = " ".join([
            f.get("airline", ""), f.get("flightNumber", ""),
            f.get("origin", ""), f.get("destination", ""),
            f.get("cabin", ""), f.get("comment", ""),
        ]).lower()
        if query_lower in haystack:
            results.append(f)
    return results[:15]


@app.route("/api/predict", methods=["GET", "POST"])
def predict():
    if request.method == "POST":
        body = request.get_json(silent=True) or {}
        duration = float(body.get("duration_min", 120))
        stops = int(body.get("num_stops", 0))
        cabin = body.get("cabin", "Economy")
        is_weekend = int(body.get("is_weekend", 0))
        sentiment_score = float(body.get("sentiment_score", 5.0))
        dep_hour = int(body.get("departure_hour", 9))

        # Simple regression-like formula
        base = 180
        base += duration * 0.8
        base += stops * 75
        if cabin == "Business":
            base *= 2.3
        elif cabin == "First":
            base *= 4.0
        base += is_weekend * 30
        base -= sentiment_score * 5
        if 6 <= dep_hour <= 9:
            base *= 1.15  # morning premium
        predicted = int(base)
        lower = int(predicted * 0.82)
        upper = int(predicted * 1.18)

        # Price tier classification
        if predicted < 100:
            price_tier = "Low"
        elif predicted <= 200:
            price_tier = "Medium"
        else:
            price_tier = "High"

        return jsonify({
            "predictedPrice": predicted,
            "price": predicted,
            "lowerBound": lower,
            "upperBound": upper,
            "sentiment": "positive" if sentiment_score > 6 else "neutral",
            "price_tier": price_tier,
        })

    # GET — predict by route + days
    origin = request.args.get("origin", "").strip().upper()
    destination = (request.args.get("destination") or request.args.get("dest", "")).strip().upper()
    days = request.args.get("days", type=int, default=30)

    if days < 1:
        return jsonify({"error": "days must be >= 1"}), 400

    flights = _all_flights()
    route_flights = [
        f for f in flights
        if f["origin"].upper() == origin and f["destination"].upper() == destination
    ]

    if route_flights:
        prices = [f["price"] for f in route_flights]
        avg = sum(prices) / len(prices)
    else:
        # Seed a consistent price for unknown routes
        seed = hashlib.md5(f"{origin}{destination}".encode()).hexdigest()
        avg = 200 + int(seed[:4], 16) % 500

    # Days-ahead discount: booking further out = cheaper
    if days >= 60:
        factor = 0.80
    elif days >= 30:
        factor = 0.90
    elif days >= 14:
        factor = 1.00
    elif days >= 7:
        factor = 1.12
    else:
        factor = 1.30  # last-minute premium

    predicted = int(avg * factor)
    lower = int(predicted * 0.82)
    upper = int(predicted * 1.18)

    # Price tier classification
    if predicted < 100:
        price_tier = "Low"
    elif predicted <= 200:
        price_tier = "Medium"
    else:
        price_tier = "High"

    # Overall route sentiment
    comments = [f.get("comment", "") for f in route_flights if f.get("comment")]
    if comments:
        sentiments = [_analyze_sentiment(c) for c in comments]
        pos = sum(1 for s in sentiments if s["sentiment"] == "positive")
        sentiment = "positive" if pos > len(sentiments) / 2 else "neutral"
    else:
        sentiment = "neutral"

    return jsonify({
        "predictedPrice": predicted,
        "price": predicted,
        "lowerBound": lower,
        "upperBound": upper,
        "sentiment": sentiment,
        "route": f"{origin} → {destination}",
        "daysAhead": days,
        "price_tier": price_tier,
    })


@app.route("/api/explore", methods=["GET"])
@app.route("/api/insights", methods=["GET"])
def explore():
    airline_filter = request.args.get("airline", "").strip()
    flights = _all_flights()

    if airline_filter:
        flights = [
            f for f in flights
            if airline_filter.lower() in f.get("airline", "").lower()
        ]

    total = len(flights)
    prices = [f["price"] for f in flights] if flights else [0]
    avg_price = int(sum(prices) / len(prices))

    # Sentiment breakdown
    sentiments = [_analyze_sentiment(f.get("comment", "")) for f in flights if f.get("comment")]
    pos = sum(1 for s in sentiments if s["sentiment"] == "positive")
    neg = sum(1 for s in sentiments if s["sentiment"] == "negative")
    neu = len(sentiments) - pos - neg
    total_s = max(len(sentiments), 1)

    overall = "positive" if pos > neg else ("negative" if neg > pos else "neutral")
    satisfaction = int((pos / total_s) * 100)

    return jsonify({
        "totalFlights": total,
        "avgPrice": avg_price,
        "sentiment": overall,
        "satisfaction": satisfaction,
        "breakdown": {
            "positive": int((pos / total_s) * 100),
            "neutral": int((neu / total_s) * 100),
            "negative": int((neg / total_s) * 100),
        },
    })


@app.route("/api/keywords", methods=["GET"])
def keywords():
    flights = _all_flights()
    kws = _extract_keywords(flights, top_n=20)
    return jsonify({"keywords": kws})


@app.route("/api/sentiment", methods=["POST"])
def sentiment():
    body = request.get_json(silent=True) or {}
    text = body.get("text", "")
    if not text.strip():
        return jsonify({"error": "No text provided"}), 400

    result = _analyze_sentiment(text)
    return jsonify(result)


@app.route("/api/flights", methods=["GET"])
def flights_list():
    flights = _all_flights()

    # Filters
    airline = request.args.get("airline", "").strip()
    origin = request.args.get("origin", "").strip().upper()
    dest = request.args.get("destination", request.args.get("dest", "")).strip().upper()
    cabin = request.args.get("cabin", "").strip()
    sort_by = request.args.get("sort", "price")
    page = request.args.get("page", 1, type=int)
    per_page = request.args.get("per_page", 20, type=int)

    if airline:
        flights = [f for f in flights if airline.lower() in f.get("airline", "").lower()]
    if origin:
        flights = [f for f in flights if f["origin"].upper() == origin]
    if dest:
        flights = [f for f in flights if f["destination"].upper() == dest]
    if cabin:
        flights = [f for f in flights if f.get("cabin", "").lower() == cabin.lower()]

    # Sort
    reverse = sort_by.startswith("-")
    key = sort_by.lstrip("-")
    flights.sort(key=lambda f: f.get(key, 0), reverse=reverse)

    # Paginate
    total = len(flights)
    start = (page - 1) * per_page
    end = start + per_page
    page_flights = flights[start:end]

    return jsonify({
        "flights": page_flights,
        "total": total,
        "page": page,
        "perPage": per_page,
        "totalPages": math.ceil(total / per_page) if per_page else 1,
    })


def _running_in_notebook():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except Exception:
        return False


# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    port = int(os.environ.get("PORT", 5000))
    print(f"\n[START] Wafrly AI API running at http://localhost:{port}")
    print(f"[DATA]  {len(_all_flights())} flight records ready\n")
    if _running_in_notebook():
        app.run(host="0.0.0.0", port=port, debug=True, use_reloader=False)
    else:
        app.run(host="0.0.0.0", port=port, debug=True)


[OK] Loaded dataset: flights_dataset_clean.xlsx  (39 records)

[START] Wafrly AI API running at http://localhost:5000
[DATA]  39 flight records ready

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.12:5000
Press CTRL+C to quit
C:\Users\Ahmed Tarek\AppData\Local\Temp\ipykernel_21348\1632439315.py:290: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",
127.0.0.1 - - [20/May/2026 03:16:41] "GET /api/health HTTP/1.1" 200 -
C:\Users\Ahmed Tarek\AppData\Local\Temp\ipykernel_21348\1632439315.py:290: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",
127.0.0.1 - - [20/May/2026 03:16:41] "GET /api/health HTTP/1.1" 200 -
127.0.0.1 - - [20/May/2026 03:16:41] "GET /api/stats HTTP/1.1" 200